# Week 4 Presentation Brief — Variation B
## 💊 Drug Pharmacokinetics: Dosing for Diabetes Management
**SCIE1500 — Semester 2, 2026 | Group 6 — presenting in Week 5**

> Work through all parts during the Week 4 lab. Your **10-minute Week 5 presentation** should cover: the problem, your model, key results, and dosing implications.

---
## 📋 Scenario

A pharmacologist is designing a dosing protocol for metformin (diabetes medication). Blood concentration after oral administration:

$$C(t) = 85\left(e^{-0.12t} - e^{-0.45t}\right)$$

where $C$ is blood concentration (μg/mL) and $t$ is hours after ingestion.

**Therapeutic constraints:**
- Minimum effective: 15 μg/mL
- Maximum safe: 100 μg/mL
- Need therapeutic levels for 12-hour dosing intervals

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Concentration profile and peak time | ~20 min |
| B | Therapeutic window analysis | ~20 min |
| C | Multiple dosing strategy | ~15 min |
| D | Rate of elimination | ~10 min |

In [ ]:
# Run first — loads libraries for this session
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
from scipy.optimize import fsolve

# Metformin pharmacokinetic parameters
D = 85     # dose factor

def C(t):
    'Blood concentration (μg/mL), t hours after ingestion.'
    return D * (np.exp(-0.12*t) - np.exp(-0.45*t))

def dC(t):
    'Rate of change of concentration (μg/mL/hour).'
    return D * (-0.12*np.exp(-0.12*t) + 0.45*np.exp(-0.45*t))

print("C(0) =", C(0), "μg/mL  (starts at zero — absorption not instant)")
print("C(1) =", round(C(1), 2), "μg/mL")
print("C(4) =", round(C(4), 2), "μg/mL")

---
## Part A: Concentration Profile (~20 min)

In [ ]:
# A.1 — Concentration at key times
print(f"{'t (h)':>6} | {'C(t) μg/mL':>12} | {'C\'(t)':>10} | Trend")
print("-" * 50)
for t in [0, 1, 2, 4, 8, 12]:
    trend = "↑ rising" if dC(t) > 0 else "↓ falling"
    print(f"{t:>6} | {C(t):>12.2f} | {dC(t):>10.3f} | {trend}")

In [ ]:
# A.2 — Find exact peak time by solving C'(t) = 0
# C'(t) = 85 * (-0.12*exp(-0.12t) + 0.45*exp(-0.45t)) = 0
# Rearranging: 0.45*exp(-0.45t) = 0.12*exp(-0.12t)
# → 0.45/0.12 = exp(0.33t) → t = ln(0.45/0.12) / 0.33

t_peak_analytic = np.log(0.45 / 0.12) / 0.33
C_peak = C(t_peak_analytic)

print(f"Peak time (analytic): t = {t_peak_analytic:.3f} hours = {t_peak_analytic*60:.0f} minutes")
print(f"Peak concentration:   C = {C_peak:.2f} μg/mL")
print(f"Safe? (< 100 μg/mL): {C_peak < 100}")

In [ ]:
# A.3 — Full concentration profile plot
t_vals = np.linspace(0, 24, 500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9))

ax1.plot(t_vals, C(t_vals), "steelblue", lw=2, label="Concentration C(t)")
ax1.axhline(15,  color="red",    ls="--", lw=1.5, label="Min therapeutic (15 μg/mL)")
ax1.axhline(100, color="orange", ls="--", lw=1.5, label="Max safe (100 μg/mL)")
ax1.fill_between(t_vals, 15, np.minimum(C(t_vals), 100),
                 where=(C(t_vals) >= 15), alpha=0.2, color="green", label="Therapeutic window")
ax1.plot(t_peak_analytic, C_peak, "ro", ms=10, label=f"Peak: t={t_peak_analytic:.2f}h, C={C_peak:.1f}")
ax1.set_ylabel("Concentration (μg/mL)")
ax1.set_title("Metformin Pharmacokinetics")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2.plot(t_vals, dC(t_vals), "purple", lw=2, label="C'(t) = dC/dt")
ax2.axhline(0, color="k", lw=0.8)
ax2.axvline(t_peak_analytic, color="red", ls=":", alpha=0.5, label=f"Peak at {t_peak_analytic:.2f}h")
ax2.set_xlabel("Time (hours)")
ax2.set_ylabel("Rate of change (μg/mL/h)")
ax2.set_title("Rate of Concentration Change")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part B: Therapeutic Window (~20 min)

In [ ]:
# B.1 — When does C(t) drop below therapeutic threshold of 15 μg/mL?
C_min = 15

# Numerical solution: find t > t_peak where C(t) = 15
def C_minus_min(t):
    return C(t) - C_min

# Search after the peak
t_end_guess = 12
t_end = fsolve(C_minus_min, t_end_guess)[0]
print(f"Therapeutic coverage ends at t = {t_end:.2f} hours")
print(f"Duration above threshold: {t_end:.2f} hours")
print(f"12-hour dosing interval sufficient? {t_end >= 12}")

✏️ **Activity B:**
1. Can patients safely use a 12-hour dosing interval? What is the concentration at t = 12 hours?
2. If coverage is insufficient, what dose multiplier $D$ is needed to ensure $C(12) = 15$?

```python
# 1. Concentration at 12 hours:
print("C(12) =", round(C(12), 3), "μg/mL")
# 2. Required dose multiplier:
D_need = 15 / (np.exp(-0.12*12) - np.exp(-0.45*12))
print("Required D:", round(D_need, 2))
print("Peak with new D:", round(D_need * (np.exp(-0.12*t_peak_analytic) - np.exp(-0.45*t_peak_analytic)), 2), "μg/mL")
```

---
## Part C: Multiple Dosing (~15 min)

In [ ]:
# C.1 — What concentration remains from first dose when second dose is taken?
C_residual = C(12)
C_peak_dose2 = C_residual + D    # residual + peak of new dose (at t=12+)

print(f"Residual from first dose at t=12h: {C_residual:.3f} μg/mL")
print(f"Immediately after second dose:     {C_peak_dose2:.2f} μg/mL")
print(f"Exceeds safe limit (100 μg/mL)?   {C_peak_dose2 > 100}")

---
## Part D: Rate of Elimination (~10 min)

In [ ]:
# D.1 — Compare elimination rate near peak vs late phase
for t_s in [2, 10]:
    rate = dC(t_s)
    pct  = rate / C(t_s) * 100
    phase = "absorption-dominant" if t_s < t_peak_analytic else "elimination-dominant"
    print(f"t = {t_s}h: C'(t) = {rate:.3f} μg/mL/h  ({pct:.1f}%/h)  [{phase}]")

---
## ✅ Presentation Checklist (Week 5, 10 minutes)

1. **Problem** (~2 min): Why does dosing protocol matter for this drug?
2. **Model** (~3 min): Double-exponential form; what each term represents (absorption vs elimination).
3. **Results** (~3 min): Peak time, therapeutic window duration, safety at 12h — show the two-panel plot.
4. **Recommendation** (~2 min): Is 12-hour dosing adequate? What dose adjustment is needed?